In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
import numpy as np
import jax
import jax.numpy as jnp
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# SAME OUTPUT MAP AS TRAINING
# ============================================================
rho_f = DTYPE(11096.0)
V_INLET_BC = DTYPE(0.4)
T_INLET = DTYPE(560.0)

PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(350.0)
TF_OUT_SCALE  = DTYPE(250.0)
U_OUT_SCALE   = DTYPE(1)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(7)
   # represent large transient pressure if needed

# =============================
# Paths
# ============================================================
theta_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/theta_sqp_pinn_cleanv2.npy"
phi_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi_correct.npy"
tfuel_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfuel.npy"
tfluid_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfluid.npy"

layer_sizes = [3, 35, 35, 35, 6]

# ============================================================
# Helpers
# ============================================================
def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]
    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size]
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.shape[0]:
        raise ValueError(f"Unused parameters remain: used {idx}, total {theta.shape[0]}")
    return params

def load_params(theta_path, layer_sizes):
    loaded = np.load(theta_path, allow_pickle=True)
    print("loaded type :", type(loaded))
    print("loaded dtype:", getattr(loaded, "dtype", None))
    print("loaded shape:", getattr(loaded, "shape", None))
    theta = jnp.array(loaded, dtype=DTYPE)
    params = unflatten_params(theta, layer_sizes)
    return params

def predict_on_rect_grid(params, xlo, xhi, ylo, yhi, tlo, thi, Nt, Nx, Ny):
    t_vals = np.linspace(float(tlo), float(thi), Nt)
    x_vals = np.linspace(float(xlo), float(xhi), Nx)
    y_vals = np.linspace(float(ylo), float(yhi), Ny)

    Tg, Xg, Yg = np.meshgrid(t_vals, x_vals, y_vals, indexing="ij")
    pts = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

    pred = np.array(mlp_apply(params, jnp.array(pts, dtype=DTYPE)))
    pred = pred.reshape(Nt, Nx, Ny, 6)
    return pred

def rel_l2(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a)
    b = b - np.mean(b)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def print_field_stats(name, arr):
    arr = np.asarray(arr)
    print(
        f"{name}: shape={arr.shape}, "
        f"min={arr.min():.6e}, max={arr.max():.6e}, "
        f"mean={arr.mean():.6e}, std={arr.std():.6e}"
    )

# ============================================================
# Load
# ============================================================
params = load_params(theta_path, layer_sizes)

print("\nLoaded params layers:", len(params))
for i, layer in enumerate(params):
    print(f"layer {i}: W{tuple(layer['W'].shape)}, b{tuple(layer['b'].shape)}")

phi_all = np.load(phi_path)
tfuel_all = np.load(tfuel_path)
tfluid_all = np.load(tfluid_path)

print("phi_all.shape   =", phi_all.shape)
print("tfuel_all.shape =", tfuel_all.shape)
print("tfluid_all.shape=", tfluid_all.shape)

phi_true   = phi_all[0] if phi_all.ndim == 4 else phi_all
tfuel_true = tfuel_all[0] if tfuel_all.ndim == 5 else tfuel_all
tfluid_true= tfluid_all[0] if tfluid_all.ndim == 5 else tfluid_all

Ts_true = tfuel_true[0]
Tf_true = tfluid_true[0]
p_true  = tfluid_true[1]
u_true  = tfluid_true[2]
v_true  = tfluid_true[3]

print("phi_true.shape   =", phi_true.shape)
print("tfuel_true.shape =", tfuel_true.shape)
print("tfluid_true.shape=", tfluid_true.shape)

# ============================================================
# Predict
# ============================================================
Nt_phi, Nx_phi, Ny_phi = phi_true.shape
pred_phi_all = predict_on_rect_grid(params, x_min, x_max, y_min, y_max, t_min, t_max, Nt_phi, Nx_phi, Ny_phi)
phi_pred = pred_phi_all[..., 0]

Nt_Ts, Nx_Ts, Ny_Ts = Ts_true.shape
pred_Ts_all = predict_on_rect_grid(params, x_min, Ls, y_min, y_max, t_min, t_max, Nt_Ts, Nx_Ts, Ny_Ts)
Ts_pred = pred_Ts_all[..., 1]

Nt_f, Nx_f, Ny_f = Tf_true.shape
pred_f_all = predict_on_rect_grid(params, Ls, x_max, y_min, y_max, t_min, t_max, Nt_f, Nx_f, Ny_f)
Tf_pred = pred_f_all[..., 5]
p_pred  = pred_f_all[..., 4]
u_pred  = pred_f_all[..., 2]
v_pred  = pred_f_all[..., 3]

print("\nPred shapes:")
print("phi_pred:", phi_pred.shape, "phi_true:", phi_true.shape)
print("Ts_pred :", Ts_pred.shape,  "Ts_true :", Ts_true.shape)
print("Tf_pred :", Tf_pred.shape,  "Tf_true :", Tf_true.shape)
print("u_pred  :", u_pred.shape,   "u_true  :", u_true.shape)
print("v_pred  :", v_pred.shape,   "v_true  :", v_true.shape)
print("p_pred  :", p_pred.shape,   "p_true  :", p_true.shape)

print("\n================ FIELD STATS ================\n")
print_field_stats("phi_pred", phi_pred)
print_field_stats("phi_true", phi_true)
print_field_stats("Ts_pred", Ts_pred)
print_field_stats("Ts_true", Ts_true)
print_field_stats("Tf_pred", Tf_pred)
print_field_stats("Tf_true", Tf_true)
print_field_stats("u_pred", u_pred)
print_field_stats("u_true", u_true)
print_field_stats("v_pred", v_pred)
print_field_stats("v_true", v_true)
print_field_stats("p_pred", p_pred)
print_field_stats("p_true", p_true)

print("\n================ Relative L2 Errors ================\n")
print("phi :", rel_l2(phi_pred, phi_true))

print("Ts  :", rel_l2(Ts_pred, Ts_true))
print("Tf  :", rel_l2(Tf_pred, Tf_true))
print("u   :", rel_l2(u_pred,  u_true))
print("v   :", rel_l2(v_pred,  v_true))
print("p(raw)      :", rel_l2(p_pred, p_true))
print("p(zero-mean):", rel_l2_zero_mean_pressure(p_pred, p_true))

loaded type : <class 'numpy.ndarray'>
loaded dtype: float64
loaded shape: (2876,)

Loaded params layers: 4
layer 0: W(3, 35), b(35,)
layer 1: W(35, 35), b(35,)
layer 2: W(35, 35), b(35,)
layer 3: W(35, 6), b(6,)
phi_all.shape   = (17, 20, 64)
tfuel_all.shape = (1, 2, 17, 8, 64)
tfluid_all.shape= (1, 4, 17, 12, 64)
phi_true.shape   = (17, 20, 64)
tfuel_true.shape = (2, 17, 8, 64)
tfluid_true.shape= (4, 17, 12, 64)

Pred shapes:
phi_pred: (17, 20, 64) phi_true: (17, 20, 64)
Ts_pred : (17, 8, 64) Ts_true : (17, 8, 64)
Tf_pred : (17, 12, 64) Tf_true : (17, 12, 64)
u_pred  : (17, 12, 64) u_true  : (17, 12, 64)
v_pred  : (17, 12, 64) v_true  : (17, 12, 64)
p_pred  : (17, 12, 64) p_true  : (17, 12, 64)

================ FIELD STATS ================

phi_pred: shape=(17, 20, 64), min=-1.517181e+00, max=5.191922e+00, mean=1.079850e+00, std=8.228145e-01
phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.391333e+

: 

In [11]:
p_pred_ct = p_pred - np.mean(p_pred, axis=(1,2), keepdims=True)
p_true_ct = p_true - np.mean(p_true, axis=(1,2), keepdims=True)

def rel_l2(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(b) + 1e-30)

print("p zero-mean per-time:", rel_l2(p_pred_ct, p_true_ct))

p zero-mean per-time: 1.003399890794477


In [15]:
def stats(name, a):
    a = np.asarray(a)
    print(f"{name}: shape={a.shape}, min={a.min():.6e}, max={a.max():.6e}, mean={a.mean():.6e}, std={a.std():.6e}")

print("\n================ FIELD STATS ================\n")
stats("phi_pred", phi_pred)
stats("phi_true", phi_true)
stats("Ts_pred", Ts_pred)
stats("Ts_true", Ts_true)
stats("Tf_pred", Tf_pred)
stats("Tf_true", Tf_true)
stats("u_pred", u_pred)
stats("u_true", u_true)
stats("v_pred", v_pred)
stats("v_true", v_true)
stats("p_pred", p_pred)
stats("p_true", p_true)

print("\n================ VELOCITY CHECKS ================\n")
print("u vs u   :", rel_l2(u_pred, u_true))
print("u vs v   :", rel_l2(u_pred, v_true))
print("u vs -u  :", rel_l2(u_pred, -u_true))
print("v vs v   :", rel_l2(v_pred, v_true))
print("v vs u   :", rel_l2(v_pred, u_true))
print("v vs -v  :", rel_l2(v_pred, -v_true))

print("\n================ PRESSURE CHECKS ================\n")
p_pred_c = p_pred - np.mean(p_pred)
p_true_c = p_true - np.mean(p_true)
print("p raw       :", rel_l2(p_pred, p_true))
print("p zero-mean :", rel_l2(p_pred_c, p_true_c))


================ FIELD STATS ================

phi_pred: shape=(17, 20, 64), min=-1.356086e+00, max=6.005833e+00, mean=3.419220e-01, std=9.057885e-01
phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.313231e+02, max=6.223336e+02, mean=5.661089e+02, std=9.167373e+00
Ts_true: shape=(17, 8, 64), min=5.600000e+02, max=8.913380e+02, mean=6.463815e+02, std=6.595224e+01
Tf_pred: shape=(17, 12, 64), min=5.592028e+02, max=5.663090e+02, mean=5.608301e+02, std=1.116534e+00
Tf_true: shape=(17, 12, 64), min=5.599427e+02, max=7.065642e+02, mean=5.736291e+02, std=2.452958e+01
u_pred: shape=(17, 12, 64), min=-2.511654e-01, max=1.159493e+00, mean=4.646365e-02, std=2.421319e-01
u_true: shape=(17, 12, 64), min=0.000000e+00, max=0.000000e+00, mean=0.000000e+00, std=0.000000e+00
v_pred: shape=(17, 12, 64), min=-5.344432e-02, max=5.373544e-01, mean=3.626856e-01, std=1.107398e-01
v_true: shape=(17, 12, 64), min=4.000000e-0

In [17]:
 u_left_true = u_true[:, 0, :]
v_left_true = v_true[:, 0, :]

u_left_pred = u_pred[:, 0, :]
v_left_pred = v_pred[:, 0, :]

print("u_left_true mean abs:", np.mean(np.abs(u_left_true)), "max abs:", np.max(np.abs(u_left_true)))
print("u_left_pred mean abs:", np.mean(np.abs(u_left_pred)), "max abs:", np.max(np.abs(u_left_pred)))

print("v_left_true mean abs:", np.mean(np.abs(v_left_true)), "max abs:", np.max(np.abs(v_left_true)))
print("v_left_pred mean abs:", np.mean(np.abs(v_left_pred)), "max abs:", np.max(np.abs(v_left_pred)))

print("u_left rel L2:", np.linalg.norm(u_left_pred - u_left_true) / (np.linalg.norm(u_left_true) + 1e-30))
print("v_left rel L2:", np.linalg.norm(v_left_pred - v_left_true) / (np.linalg.norm(v_left_true) + 1e-30))

u_left_true mean abs: 0.0 max abs: 0.0
u_left_pred mean abs: 0.09936557228977802 max abs: 1.0091057827858563
v_left_true mean abs: 0.3999999999999999 max abs: 0.4
v_left_pred mean abs: 0.3519341793020498 max abs: 0.5026801242447616
u_left rel L2: 8.121191067515975e+30
v_left rel L2: 0.3173522667151129


In [10]:
import numpy as np

# right boundary values
u_right = u_true[:, -1, :]
v_right = v_true[:, -1, :]
p_right = p_true[:, -1, :]

print("RIGHT boundary values")
print("u_right mean abs:", np.mean(np.abs(u_right)), "max abs:", np.max(np.abs(u_right)))
print("v_right mean abs:", np.mean(np.abs(v_right)), "max abs:", np.max(np.abs(v_right)))
print("p_right mean abs:", np.mean(np.abs(p_right)), "max abs:", np.max(np.abs(p_right)))

RIGHT boundary values
u_right mean abs: 0.05883472085203793 max abs: 1.0
v_right mean abs: 0.3764320733599374 max abs: 0.43804382774785144
p_right mean abs: 0.14200804697569974 max abs: 6.749538736379739


In [11]:
# use your actual fluid grid spacing in x if available
dx = 0.0114 / (12 - 1)   # if 12 grid points across x in the fluid truth

vx_right = (v_true[:, -1, :] - v_true[:, -2, :]) / dx
px_right = (p_true[:, -1, :] - p_true[:, -2, :]) / dx
ux_right = (u_true[:, -1, :] - u_true[:, -2, :]) / dx
Tfx_right = (Tf_true[:, -1, :] - Tf_true[:, -2, :]) / dx

print("\nRIGHT boundary x-derivatives")
print("vx_right mean abs:", np.mean(np.abs(vx_right)), "max abs:", np.max(np.abs(vx_right)))
print("px_right mean abs:", np.mean(np.abs(px_right)), "max abs:", np.max(np.abs(px_right)))
print("ux_right mean abs:", np.mean(np.abs(ux_right)), "max abs:", np.max(np.abs(ux_right)))
print("Tfx_right mean abs:", np.mean(np.abs(Tfx_right)), "max abs:", np.max(np.abs(Tfx_right)))


RIGHT boundary x-derivatives
vx_right mean abs: 0.5399444169064934 max abs: 9.955333705238939
px_right mean abs: 1.3141071230252523 max abs: 143.02041270663273
ux_right mean abs: 0.014976543640327392 max abs: 0.7577130786142873
Tfx_right mean abs: 182.5590890043205 max abs: 1109.8758362893595


In [12]:
u_right_mid = u_true[:, -1, 3:-3]
print("u_right_mid mean abs:", np.mean(np.abs(u_right_mid)))
print("u_right_mid max  abs:", np.max(np.abs(u_right_mid)))

u_right_mid mean abs: 0.058834558090883955
u_right_mid max  abs: 1.0


In [9]:
import numpy as np

phi_t0 = phi_true[0, :, :]
Ts_t0  = Ts_true[0, :, :]
Tf_t0  = Tf_true[0, :, :]
u_t0   = u_true[0, :, :]
v_t0   = v_true[0, :, :]

print("t=0 checks")
print("phi_t0 mean:", np.mean(phi_t0), "min:", np.min(phi_t0), "max:", np.max(phi_t0))
print("Ts_t0 mean:", np.mean(Ts_t0), "min:", np.min(Ts_t0), "max:", np.max(Ts_t0))
print("Tf_t0 mean:", np.mean(Tf_t0), "min:", np.min(Tf_t0), "max:", np.max(Tf_t0))
print("u_t0 mean:", np.mean(u_t0), "min:", np.min(u_t0), "max:", np.max(u_t0))
print("v_t0 mean:", np.mean(v_t0), "min:", np.min(v_t0), "max:", np.max(v_t0))

t=0 checks
phi_t0 mean: 1.2736294023845356 min: 0.05063449870848282 max: 1.998796676955403
Ts_t0 mean: 560.0 min: 560.0 max: 560.0
Tf_t0 mean: 560.0 min: 560.0 max: 560.0
u_t0 mean: 1.0 min: 1.0 max: 1.0
v_t0 mean: 1e-12 min: 1e-12 max: 1e-12


In [10]:
print("u_right min:", np.min(u_true[:, -1, :]))
print("u_right max:", np.max(u_true[:, -1, :]))
print("u_right first time slice:", u_true[0, -1, :10])
print("u_right middle time slice:", u_true[u_true.shape[0]//2, -1, :10])
print("u_right last time slice:", u_true[-1, -1, :10])

u_right min: -0.00046130364774520387
u_right max: 1.0
u_right first time slice: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
u_right middle time slice: [ 6.48386836e-07  1.59142714e-07 -3.38592675e-07  1.07791388e-06
 -1.30677050e-06  1.98473953e-06 -2.24415546e-06  2.84422914e-06
 -3.12661338e-06  3.63378278e-06]
u_right last time slice: [-1.81635832e-07  1.40499828e-07 -1.44593796e-07  1.01599286e-07
 -1.14715741e-07  6.97204865e-08 -9.14332138e-08  4.38977965e-08
 -7.38436535e-08  2.31611548e-08]


In [11]:
u_right_noinit = u_true[1:, -1, :]
print("u_right_noinit mean abs:", np.mean(np.abs(u_right_noinit)))
print("u_right_noinit max  abs:", np.max(np.abs(u_right_noinit)))

u_right_noinit mean abs: 1.1890905290305801e-05
u_right_noinit max  abs: 0.00046130364774520387


In [12]:
dx = 0.0114 / (12 - 1)
sl = slice(3, -3)

vx_right_mid  = (v_true[1:, -1, sl]  - v_true[1:, -2, sl])  / dx

px_right_mid  = (p_true[1:, -1, sl]  - p_true[1:, -2, sl])  / dx
Tfx_right_mid = (Tf_true[1:, -1, sl] - Tf_true[1:, -2, sl]) / dx

print("vx_right_mid mean abs:", np.mean(np.abs(vx_right_mid)), "max abs:", np.max(np.abs(vx_right_mid)))
print("px_right_mid mean abs:", np.mean(np.abs(px_right_mid)), "max abs:", np.max(np.abs(px_right_mid)))
print("Tfx_right_mid mean abs:", np.mean(np.abs(Tfx_right_mid)), "max abs:", np.max(np.abs(Tfx_right_mid)))

vx_right_mid mean abs: 0.6001001655876222 max abs: 9.955333705238939
px_right_mid mean abs: 1.3174375723520315 max abs: 143.02041270663273
Tfx_right_mid mean abs: 192.36031733680252 max abs: 1084.2745712122785


In [13]:
dx = 0.0114 / (12 - 1)
sl = slice(3, -3)

u_right_mid   = u_true[1:, -1, sl]
vx_right_mid  = (v_true[1:, -1, sl]  - v_true[1:, -2, sl])  / dx
px_right_mid  = (p_true[1:, -1, sl]  - p_true[1:, -2, sl])  / dx
Tfx_right_mid = (Tf_true[1:, -1, sl] - Tf_true[1:, -2, sl]) / dx

print("u_right_mid mean abs:",   np.mean(np.abs(u_right_mid)),   "max abs:", np.max(np.abs(u_right_mid)))
print("vx_right_mid mean abs:",  np.mean(np.abs(vx_right_mid)),  "max abs:", np.max(np.abs(vx_right_mid)))
print("px_right_mid mean abs:",  np.mean(np.abs(px_right_mid)),  "max abs:", np.max(np.abs(px_right_mid)))
print("Tfx_right_mid mean abs:", np.mean(np.abs(Tfx_right_mid)), "max abs:", np.max(np.abs(Tfx_right_mid)))

u_right_mid mean abs: 1.1717971564207714e-05 max abs: 0.00039322087466922757
vx_right_mid mean abs: 0.6001001655876222 max abs: 9.955333705238939
px_right_mid mean abs: 1.3174375723520315 max abs: 143.02041270663273
Tfx_right_mid mean abs: 192.36031733680252 max abs: 1084.2745712122785


In [14]:
dx = 0.0114 / (12 - 1)
sl = slice(3, -3)

vx_last = (v_true[1:, -1, sl] - v_true[1:, -2, sl]) / dx
vx_prev = (v_true[1:, -2, sl] - v_true[1:, -3, sl]) / dx

px_last = (p_true[1:, -1, sl] - p_true[1:, -2, sl]) / dx
px_prev = (p_true[1:, -2, sl] - p_true[1:, -3, sl]) / dx

Tfx_last = (Tf_true[1:, -1, sl] - Tf_true[1:, -2, sl]) / dx
Tfx_prev = (Tf_true[1:, -2, sl] - Tf_true[1:, -3, sl]) / dx

def stats(name, a):
    print(name, "mean abs:", np.mean(np.abs(a)), "max abs:", np.max(np.abs(a)))

stats("vx_last", vx_last)
stats("vx_prev", vx_prev)
stats("px_last", px_last)
stats("px_prev", px_prev)
stats("Tfx_last", Tfx_last)
stats("Tfx_prev", Tfx_prev)

vx_last mean abs: 0.6001001655876222 max abs: 9.955333705238939
vx_prev mean abs: 0.6173036971365657 max abs: 8.914788011141516
px_last mean abs: 1.3174375723520315 max abs: 143.02041270663273
px_prev mean abs: 2.4316086134633954 max abs: 245.13745949219256
Tfx_last mean abs: 192.36031733680252 max abs: 1084.2745712122785
Tfx_prev mean abs: 424.7463130067814 max abs: 2241.7391141736807


In [15]:
# phi_true shape assumed: (Nt, Nx, Ny)

print("RIGHT phi")
phi_right = phi_true[:, -1, :]
print("mean abs:", np.mean(np.abs(phi_right)))
print("max abs :", np.max(np.abs(phi_right)))
print("min/max :", np.min(phi_right), np.max(phi_right))

print("\nBOTTOM phi")
phi_bottom = phi_true[:, :, 0]
print("mean abs error to 0.5:", np.mean(np.abs(phi_bottom - 0.5)))
print("max abs error  to 0.5:", np.max(np.abs(phi_bottom - 0.5)))
print("min/max :", np.min(phi_bottom), np.max(phi_bottom))

print("\nTOP phi")
phi_top = phi_true[:, :, -1]
print("mean abs error to 0.5:", np.mean(np.abs(phi_top - 0.5)))
print("max abs error  to 0.5:", np.max(np.abs(phi_top - 0.5)))
print("min/max :", np.min(phi_top), np.max(phi_top))

RIGHT phi
mean abs: 0.1759052020626532
max abs : 1.998796676955403
min/max : 0.019286873502372324 1.998796676955403

BOTTOM phi
mean abs error to 0.5: 0.4284539499150004
max abs error  to 0.5: 1.1646823997930156
min/max : 0.05063449870848282 1.6646823997930156

TOP phi
mean abs error to 0.5: 0.18309981366477326
max abs error  to 0.5: 0.44936550129151653
min/max : 0.05063449870848349 0.8793695867465514


In [16]:
print("phi_true.shape =", phi_true.shape)

# assume phi_true is 3D
a, b, c = phi_true.shape
print("axis sizes:", a, b, c)

def stats(name, arr):
    print(name)
    print("  shape   :", arr.shape)
    print("  mean abs:", np.mean(np.abs(arr)))
    print("  min/max :", np.min(arr), np.max(arr))

# all six outer slices
stats("axis1 first  phi_true[0,:,:]",   phi_true[0, :, :])
stats("axis1 last   phi_true[-1,:,:]",  phi_true[-1, :, :])

stats("axis2 first  phi_true[:,0,:]",   phi_true[:, 0, :])
stats("axis2 last   phi_true[:,-1,:]",  phi_true[:, -1, :])

stats("axis3 first  phi_true[:,:,0]",   phi_true[:, :, 0])
stats("axis3 last   phi_true[:,:,-1]",  phi_true[:, :, -1])

phi_true.shape = (17, 20, 64)
axis sizes: 17 20 64
axis1 first  phi_true[0,:,:]
  shape   : (20, 64)
  mean abs: 1.2736294023845356
  min/max : 0.05063449870848282 1.998796676955403
axis1 last   phi_true[-1,:,:]
  shape   : (20, 64)
  mean abs: 2.7600419738549364
  min/max : 0.04744169422884767 10.483047280042607
axis2 first  phi_true[:,0,:]
  shape   : (17, 64)
  mean abs: 3.487378040282388
  min/max : 0.05063449870848282 10.483047280042607
axis2 last   phi_true[:,-1,:]
  shape   : (17, 64)
  mean abs: 0.1759052020626532
  min/max : 0.019286873502372324 1.998796676955403
axis3 first  phi_true[:,:,0]
  shape   : (17, 20)
  mean abs: 0.8020025868223284
  min/max : 0.05063449870848282 1.6646823997930156
axis3 last   phi_true[:,:,-1]
  shape   : (17, 20)
  mean abs: 0.5493111143768257
  min/max : 0.05063449870848349 0.8793695867465514


In [17]:
print("phi_true.shape =", phi_true.shape)

# assume phi_true is 3D
a, b, c = phi_true.shape
print("axis sizes:", a, b, c)

def stats(name, arr):
    print(name)
    print("  shape   :", arr.shape)
    print("  mean abs:", np.mean(np.abs(arr)))
    print("  min/max :", np.min(arr), np.max(arr))

# all six outer slices
stats("axis1 first  phi_true[0,:,:]",   phi_true[0, :, :])
stats("axis1 last   phi_true[-1,:,:]",  phi_true[-1, :, :])

stats("axis2 first  phi_true[:,0,:]",   phi_true[:, 0, :])
stats("axis2 last   phi_true[:,-1,:]",  phi_true[:, -1, :])

stats("axis3 first  phi_true[:,:,0]",   phi_true[:, :, 0])
stats("axis3 last   phi_true[:,:,-1]",  phi_true[:, :, -1])

phi_true.shape = (17, 20, 64)
axis sizes: 17 20 64
axis1 first  phi_true[0,:,:]
  shape   : (20, 64)
  mean abs: 1.2736294023845356
  min/max : 0.05063449870848282 1.998796676955403
axis1 last   phi_true[-1,:,:]
  shape   : (20, 64)
  mean abs: 2.7600419738549364
  min/max : 0.04744169422884767 10.483047280042607
axis2 first  phi_true[:,0,:]
  shape   : (17, 64)
  mean abs: 3.487378040282388
  min/max : 0.05063449870848282 10.483047280042607
axis2 last   phi_true[:,-1,:]
  shape   : (17, 64)
  mean abs: 0.1759052020626532
  min/max : 0.019286873502372324 1.998796676955403
axis3 first  phi_true[:,:,0]
  shape   : (17, 20)
  mean abs: 0.8020025868223284
  min/max : 0.05063449870848282 1.6646823997930156
axis3 last   phi_true[:,:,-1]
  shape   : (17, 20)
  mean abs: 0.5493111143768257
  min/max : 0.05063449870848349 0.8793695867465514


In [18]:
print("phi_true.shape =", phi_true.shape)

# assume phi_true is 3D
a, b, c = phi_true.shape
print("axis sizes:", a, b, c)

def stats(name, arr):
    print(name)
    print("  shape   :", arr.shape)
    print("  mean abs:", np.mean(np.abs(arr)))
    print("  min/max :", np.min(arr), np.max(arr))

# all six outer slices
stats("axis1 first  phi_true[0,:,:]",   phi_true[0, :, :])
stats("axis1 last   phi_true[-1,:,:]",  phi_true[-1, :, :])

stats("axis2 first  phi_true[:,0,:]",   phi_true[:, 0, :])
stats("axis2 last   phi_true[:,-1,:]",  phi_true[:, -1, :])

stats("axis3 first  phi_true[:,:,0]",   phi_true[:, :, 0])
stats("axis3 last   phi_true[:,:,-1]",  phi_true[:, :, -1])

phi_true.shape = (17, 20, 64)
axis sizes: 17 20 64
axis1 first  phi_true[0,:,:]
  shape   : (20, 64)
  mean abs: 1.2736294023845356
  min/max : 0.05063449870848282 1.998796676955403
axis1 last   phi_true[-1,:,:]
  shape   : (20, 64)
  mean abs: 2.7600419738549364
  min/max : 0.04744169422884767 10.483047280042607
axis2 first  phi_true[:,0,:]
  shape   : (17, 64)
  mean abs: 3.487378040282388
  min/max : 0.05063449870848282 10.483047280042607
axis2 last   phi_true[:,-1,:]
  shape   : (17, 64)
  mean abs: 0.1759052020626532
  min/max : 0.019286873502372324 1.998796676955403
axis3 first  phi_true[:,:,0]
  shape   : (17, 20)
  mean abs: 0.8020025868223284
  min/max : 0.05063449870848282 1.6646823997930156
axis3 last   phi_true[:,:,-1]
  shape   : (17, 20)
  mean abs: 0.5493111143768257
  min/max : 0.05063449870848349 0.8793695867465514


In [19]:
# candidate right/left
print("left candidate mean abs:", np.mean(np.abs(phi_true[:, 0, :])))
print("right candidate mean abs:", np.mean(np.abs(phi_true[:, -1, :])))
print("left candidate min/max:", np.min(phi_true[:, 0, :]), np.max(phi_true[:, 0, :]))
print("right candidate min/max:", np.min(phi_true[:, -1, :]), np.max(phi_true[:, -1, :]))

# candidate bottom/top
print("bottom candidate min/max:", np.min(phi_true[:, :, 0]), np.max(phi_true[:, :, 0]))
print("top candidate min/max   :", np.min(phi_true[:, :, -1]), np.max(phi_true[:, :, -1]))


left candidate mean abs: 3.487378040282388
right candidate mean abs: 0.1759052020626532
left candidate min/max: 0.05063449870848282 10.483047280042607
right candidate min/max: 0.019286873502372324 1.998796676955403
bottom candidate min/max: 0.05063449870848282 1.6646823997930156
top candidate min/max   : 0.05063449870848349 0.8793695867465514


nft_phi shape: (1, 17, 20, 64) min/max: -33.26056442378389 126.2331917444798
phiBC_to_phi shape: (1, 65, 16) min/max: 0.5 10.57847754161336
nft_phi first slice mean abs: 52.23436246774514
phiBC_to_phi first slice mean abs: 3.5702236321047187


ValueError: operands could not be broadcast together with shapes (1,17,20,64) (1,65,16) 

In [18]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import jax
import jax.numpy as jnp

# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# SAME OUTPUT MAP AS TRAINING
# ============================================================
T_INLET = DTYPE(560.0)

PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(350.0)
TF_OUT_SCALE  = DTYPE(250.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(7.0)   # make sure this matches training

# ============================================================
# Paths
# ============================================================
theta_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/theta_sqp_pinn_clean.npy"
phi_path    = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi_correct.npy"
tfuel_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfuel.npy"
tfluid_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfluid.npy"

layer_sizes = [3, 30, 30, 30, 6]

# ============================================================
# Helpers
# ============================================================
def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]
    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size]
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.shape[0]:
        raise ValueError(f"Unused parameters remain: used {idx}, total {theta.shape[0]}")
    return params

def load_params(theta_path, layer_sizes):
    loaded = np.load(theta_path, allow_pickle=True)
    print("loaded type :", type(loaded))
    print("loaded dtype:", getattr(loaded, "dtype", None))
    print("loaded shape:", getattr(loaded, "shape", None))
    theta = jnp.array(loaded, dtype=DTYPE)
    params = unflatten_params(theta, layer_sizes)
    return params

def centers(lo, hi, N):
    d = (float(hi) - float(lo)) / N
    return np.linspace(float(lo) + 0.5 * d, float(hi) - 0.5 * d, N)

def predict_on_rect_grid_centers(params, xlo, xhi, ylo, yhi, tlo, thi, Nt, Nx, Ny):
    # time stays at saved times / uniform times; spatial points use cell centers
    t_vals = np.linspace(float(tlo), float(thi), Nt)
    x_vals = centers(xlo, xhi, Nx)
    y_vals = centers(ylo, yhi, Ny)

    Tg, Xg, Yg = np.meshgrid(t_vals, x_vals, y_vals, indexing="ij")
    pts = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

    pred = np.array(mlp_apply(params, jnp.array(pts, dtype=DTYPE)))
    pred = pred.reshape(Nt, Nx, Ny, 6)
    return pred

def rel_l2(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a)
    b = b - np.mean(b)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure_per_time(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a, axis=(1, 2), keepdims=True)
    b = b - np.mean(b, axis=(1, 2), keepdims=True)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def print_field_stats(name, arr):
    arr = np.asarray(arr)
    print(
        f"{name}: shape={arr.shape}, "
        f"min={arr.min():.6e}, max={arr.max():.6e}, "
        f"mean={arr.mean():.6e}, std={arr.std():.6e}"
    )

# ============================================================
# Load
# ============================================================
params = load_params(theta_path, layer_sizes)

print("\nLoaded params layers:", len(params))
for i, layer in enumerate(params):
    print(f"layer {i}: W{tuple(layer['W'].shape)}, b{tuple(layer['b'].shape)}")

phi_all   = np.load(phi_path)
tfuel_all = np.load(tfuel_path)
tfluid_all= np.load(tfluid_path)

print("phi_all.shape   =", phi_all.shape)
print("tfuel_all.shape =", tfuel_all.shape)
print("tfluid_all.shape=", tfluid_all.shape)

phi_true    = phi_all[0] if phi_all.ndim == 4 else phi_all
tfuel_true  = tfuel_all[0] if tfuel_all.ndim == 5 else tfuel_all
tfluid_true = tfluid_all[0] if tfluid_all.ndim == 5 else tfluid_all

Ts_true = tfuel_true[0]
Tf_true = tfluid_true[0]
p_true  = tfluid_true[1]
u_true  = tfluid_true[2]
v_true  = tfluid_true[3]

print("phi_true.shape   =", phi_true.shape)
print("tfuel_true.shape =", tfuel_true.shape)
print("tfluid_true.shape=", tfluid_true.shape)

# ============================================================
# Predict on CELL CENTERS
# ============================================================
Nt_phi, Nx_phi, Ny_phi = phi_true.shape
pred_phi_all = predict_on_rect_grid_centers(
    params, x_min, x_max, y_min, y_max, t_min, t_max, Nt_phi, Nx_phi, Ny_phi
)
phi_pred = pred_phi_all[..., 0]

Nt_Ts, Nx_Ts, Ny_Ts = Ts_true.shape
pred_Ts_all = predict_on_rect_grid_centers(
    params, x_min, Ls, y_min, y_max, t_min, t_max, Nt_Ts, Nx_Ts, Ny_Ts
)
Ts_pred = pred_Ts_all[..., 1]

Nt_f, Nx_f, Ny_f = Tf_true.shape
pred_f_all = predict_on_rect_grid_centers(
    params, Ls, x_max, y_min, y_max, t_min, t_max, Nt_f, Nx_f, Ny_f
)
Tf_pred = pred_f_all[..., 5]
p_pred  = pred_f_all[..., 4]
u_pred  = pred_f_all[..., 2]
v_pred  = pred_f_all[..., 3]

print("\nPred shapes:")
print("phi_pred:", phi_pred.shape, "phi_true:", phi_true.shape)
print("Ts_pred :", Ts_pred.shape,  "Ts_true :", Ts_true.shape)
print("Tf_pred :", Tf_pred.shape,  "Tf_true :", Tf_true.shape)
print("u_pred  :", u_pred.shape,   "u_true  :", u_true.shape)
print("v_pred  :", v_pred.shape,   "v_true  :", v_true.shape)
print("p_pred  :", p_pred.shape,   "p_true  :", p_true.shape)

print("\n================ FIELD STATS ================\n")
print_field_stats("phi_pred", phi_pred)
print_field_stats("phi_true", phi_true)
print_field_stats("Ts_pred", Ts_pred)
print_field_stats("Ts_true", Ts_true)
print_field_stats("Tf_pred", Tf_pred)
print_field_stats("Tf_true", Tf_true)
print_field_stats("u_pred", u_pred)
print_field_stats("u_true", u_true)
print_field_stats("v_pred", v_pred)
print_field_stats("v_true", v_true)
print_field_stats("p_pred", p_pred)
print_field_stats("p_true", p_true)

print("\n================ Relative L2 Errors ================\n")
print("phi :", rel_l2(phi_pred, phi_true))
print("Ts  :", rel_l2(Ts_pred, Ts_true))
print("Tf  :", rel_l2(Tf_pred, Tf_true))
print("u   :", rel_l2(u_pred,  u_true))
print("v   :", rel_l2(v_pred,  v_true))
print("p(raw)                :", rel_l2(p_pred, p_true))
print("p(zero-mean global)   :", rel_l2_zero_mean_pressure(p_pred, p_true))
print("p(zero-mean per-time) :", rel_l2_zero_mean_pressure_per_time(p_pred, p_true))

loaded type : <class 'numpy.ndarray'>
loaded dtype: float64
loaded shape: (2166,)

Loaded params layers: 4
layer 0: W(3, 30), b(30,)
layer 1: W(30, 30), b(30,)
layer 2: W(30, 30), b(30,)
layer 3: W(30, 6), b(6,)
phi_all.shape   = (17, 20, 64)
tfuel_all.shape = (1, 2, 17, 8, 64)
tfluid_all.shape= (1, 4, 17, 12, 64)
phi_true.shape   = (17, 20, 64)
tfuel_true.shape = (2, 17, 8, 64)
tfluid_true.shape= (4, 17, 12, 64)

Pred shapes:
phi_pred: (17, 20, 64) phi_true: (17, 20, 64)
Ts_pred : (17, 8, 64) Ts_true : (17, 8, 64)
Tf_pred : (17, 12, 64) Tf_true : (17, 12, 64)
u_pred  : (17, 12, 64) u_true  : (17, 12, 64)
v_pred  : (17, 12, 64) v_true  : (17, 12, 64)
p_pred  : (17, 12, 64) p_true  : (17, 12, 64)

================ FIELD STATS ================

phi_pred: shape=(17, 20, 64), min=-1.509795e+00, max=5.750107e+00, mean=8.102537e-01, std=1.029903e+00
phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.393921e+

In [19]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import numpy as np
import jax
import jax.numpy as jnp

# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# Geometry / time
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

# ============================================================
# SAME OUTPUT MAP AS TRAINING
# ============================================================
T_INLET = DTYPE(560.0)

PHI_OUT_SCALE = DTYPE(15.0)
TS_OUT_SCALE  = DTYPE(350.0)
TF_OUT_SCALE  = DTYPE(250.0)
U_OUT_SCALE   = DTYPE(1.0)
V_OUT_SCALE   = DTYPE(0.5)
P_OUT_SCALE   = DTYPE(7.0)   # make sure this matches training

# ============================================================
# Paths
# ============================================================
theta_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/theta_sqp_pinn_clean.npy"
phi_path    = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_phi_correct.npy"
tfuel_path  = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfuel.npy"
tfluid_path = "/projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/moose/output/nft_Tfluid.npy"

layer_sizes = [3, 30, 30, 30, 6]

# ============================================================
# Helpers
# ============================================================
def normalize_xyt(X):
    x = X[:, 0:1]
    y = X[:, 1:2]
    t = X[:, 2:3]
    x_n = 2.0 * (x - x_min) / (x_max - x_min + EPS) - 1.0
    y_n = 2.0 * (y - y_min) / (y_max - y_min + EPS) - 1.0
    t_n = 2.0 * (t - t_min) / (t_max - t_min + EPS) - 1.0
    return jnp.concatenate([x_n, y_n, t_n], axis=1)

def mlp_apply(params, X):
    h = normalize_xyt(X)
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_OUT_SCALE * phi_hat
    Ts  = T_INLET + TS_OUT_SCALE * Ts_hat
    u   = U_OUT_SCALE * u_hat
    v   = V_OUT_SCALE * v_hat
    p   = P_OUT_SCALE * p_hat
    Tf  = T_INLET + TF_OUT_SCALE * Tf_hat

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def unflatten_params(theta, layer_sizes):
    params = []
    idx = 0
    for m, n in zip(layer_sizes[:-1], layer_sizes[1:]):
        w_size = m * n
        b_size = n
        W = theta[idx:idx + w_size].reshape((m, n))
        idx += w_size
        b = theta[idx:idx + b_size]
        idx += b_size
        params.append({"W": W, "b": b})
    if idx != theta.shape[0]:
        raise ValueError(f"Unused parameters remain: used {idx}, total {theta.shape[0]}")
    return params

def load_params(theta_path, layer_sizes):
    loaded = np.load(theta_path, allow_pickle=True)
    print("loaded type :", type(loaded))
    print("loaded dtype:", getattr(loaded, "dtype", None))
    print("loaded shape:", getattr(loaded, "shape", None))
    theta = jnp.array(loaded, dtype=DTYPE)
    params = unflatten_params(theta, layer_sizes)
    return params

def centers(lo, hi, N):
    d = (float(hi) - float(lo)) / N
    return np.linspace(float(lo) + 0.5 * d, float(hi) - 0.5 * d, N)

def predict_on_rect_grid_centers(params, xlo, xhi, ylo, yhi, tlo, thi, Nt, Nx, Ny):
    # time stays at saved times / uniform times; spatial points use cell centers
    t_vals = np.linspace(float(tlo), float(thi), Nt)
    x_vals = centers(xlo, xhi, Nx)
    y_vals = centers(ylo, yhi, Ny)

    Tg, Xg, Yg = np.meshgrid(t_vals, x_vals, y_vals, indexing="ij")
    pts = np.stack([Xg.ravel(), Yg.ravel(), Tg.ravel()], axis=1)

    pred = np.array(mlp_apply(params, jnp.array(pts, dtype=DTYPE)))
    pred = pred.reshape(Nt, Nx, Ny, 6)
    return pred

def rel_l2(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a)
    b = b - np.mean(b)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def rel_l2_zero_mean_pressure_per_time(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    a = a - np.mean(a, axis=(1, 2), keepdims=True)
    b = b - np.mean(b, axis=(1, 2), keepdims=True)
    return np.sqrt(np.mean((a - b) ** 2) / (np.mean(b ** 2) + 1e-30))

def print_field_stats(name, arr):
    arr = np.asarray(arr)
    print(
        f"{name}: shape={arr.shape}, "
        f"min={arr.min():.6e}, max={arr.max():.6e}, "
        f"mean={arr.mean():.6e}, std={arr.std():.6e}"
    )

# ============================================================
# Load
# ============================================================
params = load_params(theta_path, layer_sizes)

print("\nLoaded params layers:", len(params))
for i, layer in enumerate(params):
    print(f"layer {i}: W{tuple(layer['W'].shape)}, b{tuple(layer['b'].shape)}")

phi_all   = np.load(phi_path)
tfuel_all = np.load(tfuel_path)
tfluid_all= np.load(tfluid_path)

print("phi_all.shape   =", phi_all.shape)
print("tfuel_all.shape =", tfuel_all.shape)
print("tfluid_all.shape=", tfluid_all.shape)

phi_true    = phi_all[0] if phi_all.ndim == 4 else phi_all
tfuel_true  = tfuel_all[0] if tfuel_all.ndim == 5 else tfuel_all
tfluid_true = tfluid_all[0] if tfluid_all.ndim == 5 else tfluid_all

Ts_true = tfuel_true[0]
Tf_true = tfluid_true[0]
p_true  = tfluid_true[1]
u_true  = tfluid_true[2]
v_true  = tfluid_true[3]

print("phi_true.shape   =", phi_true.shape)
print("tfuel_true.shape =", tfuel_true.shape)
print("tfluid_true.shape=", tfluid_true.shape)

# ============================================================
# Predict on CELL CENTERS
# ============================================================
Nt_phi, Nx_phi, Ny_phi = phi_true.shape
pred_phi_all = predict_on_rect_grid_centers(
    params, x_min, x_max, y_min, y_max, t_min, t_max, Nt_phi, Nx_phi, Ny_phi
)
phi_pred = pred_phi_all[..., 0]

Nt_Ts, Nx_Ts, Ny_Ts = Ts_true.shape
pred_Ts_all = predict_on_rect_grid_centers(
    params, x_min, Ls, y_min, y_max, t_min, t_max, Nt_Ts, Nx_Ts, Ny_Ts
)
Ts_pred = pred_Ts_all[..., 1]

Nt_f, Nx_f, Ny_f = Tf_true.shape
pred_f_all = predict_on_rect_grid_centers(
    params, Ls, x_max, y_min, y_max, t_min, t_max, Nt_f, Nx_f, Ny_f
)
Tf_pred = pred_f_all[..., 5]
p_pred  = pred_f_all[..., 4]
u_pred  = pred_f_all[..., 2]
v_pred  = pred_f_all[..., 3]

print("\nPred shapes:")
print("phi_pred:", phi_pred.shape, "phi_true:", phi_true.shape)
print("Ts_pred :", Ts_pred.shape,  "Ts_true :", Ts_true.shape)
print("Tf_pred :", Tf_pred.shape,  "Tf_true :", Tf_true.shape)
print("u_pred  :", u_pred.shape,   "u_true  :", u_true.shape)
print("v_pred  :", v_pred.shape,   "v_true  :", v_true.shape)
print("p_pred  :", p_pred.shape,   "p_true  :", p_true.shape)

print("\n================ FIELD STATS ================\n")
print_field_stats("phi_pred", phi_pred)
print_field_stats("phi_true", phi_true)
print_field_stats("Ts_pred", Ts_pred)
print_field_stats("Ts_true", Ts_true)
print_field_stats("Tf_pred", Tf_pred)
print_field_stats("Tf_true", Tf_true)
print_field_stats("u_pred", u_pred)
print_field_stats("u_true", u_true)
print_field_stats("v_pred", v_pred)
print_field_stats("v_true", v_true)
print_field_stats("p_pred", p_pred)
print_field_stats("p_true", p_true)

print("\n================ Relative L2 Errors ================\n")
print("phi :", rel_l2(phi_pred, phi_true))
print("Ts  :", rel_l2(Ts_pred, Ts_true))
print("Tf  :", rel_l2(Tf_pred, Tf_true))
print("u   :", rel_l2(u_pred,  u_true))
print("v   :", rel_l2(v_pred,  v_true))
print("p(raw)                :", rel_l2(p_pred, p_true))
print("p(zero-mean global)   :", rel_l2_zero_mean_pressure(p_pred, p_true))
print("p(zero-mean per-time) :", rel_l2_zero_mean_pressure_per_time(p_pred, p_true))

loaded type : <class 'numpy.ndarray'>
loaded dtype: float64
loaded shape: (2166,)

Loaded params layers: 4
layer 0: W(3, 30), b(30,)
layer 1: W(30, 30), b(30,)
layer 2: W(30, 30), b(30,)
layer 3: W(30, 6), b(6,)
phi_all.shape   = (17, 20, 64)
tfuel_all.shape = (1, 2, 17, 8, 64)
tfluid_all.shape= (1, 4, 17, 12, 64)
phi_true.shape   = (17, 20, 64)
tfuel_true.shape = (2, 17, 8, 64)
tfluid_true.shape= (4, 17, 12, 64)

Pred shapes:
phi_pred: (17, 20, 64) phi_true: (17, 20, 64)
Ts_pred : (17, 8, 64) Ts_true : (17, 8, 64)
Tf_pred : (17, 12, 64) Tf_true : (17, 12, 64)
u_pred  : (17, 12, 64) u_true  : (17, 12, 64)
v_pred  : (17, 12, 64) v_true  : (17, 12, 64)
p_pred  : (17, 12, 64) p_true  : (17, 12, 64)

================ FIELD STATS ================

phi_pred: shape=(17, 20, 64), min=-1.509795e+00, max=5.750107e+00, mean=8.102537e-01, std=1.029903e+00
phi_true: shape=(17, 20, 64), min=1.928687e-02, max=1.048305e+01, mean=2.031849e+00, std=1.688021e+00
Ts_pred: shape=(17, 8, 64), min=5.393921e+